# RoPE 
### RoFormer: Enhanced Transformer with Rotary Position Embedding

- 절대위치는 회전 행렬로 인코딩하되, 그 결과 Q·K 내적을 계산하면 자동으로 상대위치 정보만 남게 만들기

In [1]:
import torch
import torch.nn as nn

In [4]:
class RotaryPositionalEmbedding(nn.Module):
    def __init__(self, head_dim: int, max_seq_len: int = 4096, base: float = 10000.0):
        super().__init__()

        self.head_dim = head_dim
        self.max_seq_len = max_seq_len

        theta = 1.0 / (base ** (torch.arange(0, head_dim, 2).float() / head_dim))
        self.register_buffer("theta", theta, persistent=False)

        self._build_cache(max_seq_len)

    def _build_cache(self, seq_len: int):
        m = torch.arange(seq_len, dtype=self.theta.dtype, device=self.theta.device)

        angles = torch.einsum(
            "i, j -> ij", m, self.theta
        )

        angles_interleaved = angles.repeat_interleave(
            2, dim=-1
        )

        self.register_buffer("cos_cached", angles_interleaved.cos(), persistent=False)
        self.register_buffer("sin_cached", angles_interleaved.sin(), persistent=False)

    @staticmethod
    def rotate_interleaved(x: torch.Tensor) -> torch.Tensor:
        x_even = x[..., 0::2]
        x_odd = x[..., 1::2]
        rotated = torch.stack((-x_odd, x_even), dim=-1)
        return rotated.flatten(
            start_dim=-2
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        seq_len = x.shape[1]

        if seq_len > self.max_seq_len:
            self._build_cache(seq_len)
            self.max_seq_len = seq_len

        cos = self.cos_cached[:seq_len].to(x.dtype)
        sin = self.sin_cached[:seq_len].to(x.dtype)

        cos = cos[None, :, None, :]
        sin = sin[None, :, None, :]

        return x * cos + self.rotate_interleaved(x) * sin

### 검증 셀 (1) : norm 보존 확인
- 회전행렬은 orthogonal matrix라서 회전 전후고 백터 크기(L2 norm)은 변하면 안된다.  

$$
R_{\Theta,m}x = x \odot \cos(m\Theta) + \text{rotate\_interleaved}(x) \odot \sin(m\Theta)
$$

In [5]:
# 1. head_dim=64로 RoPE 인스턴스 만들기 
rope = RotaryPositionalEmbedding(head_dim=64, max_seq_len=128)

# 2. 랜덤 입력 텐서 만들기 
# shape는 (batch, seq_len, head_dim) 예 : (1, 10, 64)
x = torch.randn(1 ,10 ,64 )

# 3. rope에 x를 통과시키기
x_rotated = rope(x)

# 4. 각각 norm 구하기 (마지막 차원 기준)
norm_before = x.norm(dim=-1)
norm_after = x_rotated.norm(dim=-1)

# 5. 두 값이 거의 같은지 비교
print(torch.allclose(norm_before, norm_after, atol=1e-5))

True


-> True의 의미 : RoPE로 회전시키기 전 벡터의 크기 (L2norm)와, 회전시킨 후 벡터의 크기가 거의 똑같다.

### 검증 셀 (2) : 상대위치 성질 확인 
- 절대 위치가 달라도 상대거리가 같으면 내적이 같아야 한다. 

In [6]:
head_dim = 64
rope = RotaryPositionalEmbedding(head_dim=head_dim, max_seq_len=128)

# q, k는 각각 (batch=1, 1개 토큰, head_dim) 짜리 랜덤 벡터 하나씩
q = torch.randn(1, 1, head_dim)
k = torch.randn(1, 1, head_dim)

def apply_rope_at(x, pos, rope):
    # rope.cos_cached, rope.sin_cached 에서 pos번째 위치 하나만 뽑아쓰기
    cos = rope.cos_cached[pos].to(x.dtype)  # shape: (head_dim,)
    sin = rope.sin_cached[pos].to(x.dtype)
    return x * cos + rope.rotate_interleaved(x) * sin

# 케이스 A: 위치 2, 5 (상대거리 3)
q_a = apply_rope_at(q, 2, rope)
k_a = apply_rope_at(k, 5, rope)
score_a = (q_a * k_a).sum(dim=-1)   # 내적

# 케이스 B: 위치 10, 13 (상대거리 3)
q_b = apply_rope_at(q, 10, rope)
k_b = apply_rope_at(k, 13, rope)
score_b = (q_b * k_b).sum(dim=-1)

print(score_a)
print(score_b)
print(torch.allclose(score_a, score_b, atol=1e-4))

tensor([[0.7183]])
tensor([[0.7183]])
True


-> True의 의미 : 절대위치가 뭐든 상관 없이, 상대거리만 같으면 attention score가 똑같이 나온다.